In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

RANDOM_STATE = 42


def compute_group_metrics(df, group_col):
    results = []

    for group_value, g in df.groupby(group_col):
        row = {
            group_col: group_value,
            "n": len(g),
            "accuracy": accuracy_score(g["y_true"], g["y_pred"]),
            "positive_rate": g["y_pred"].mean()
        }

        if g["y_true"].nunique() < 2:
            row["auc"] = np.nan
            row["log_loss"] = np.nan
        else:
            row["auc"] = roc_auc_score(g["y_true"], g["y_prob"])
            row["log_loss"] = log_loss(g["y_true"], g["y_prob"])

        results.append(row)

    return pd.DataFrame(results)


def disparity_summary(df, group_col, metric_col):
    temp = df[[group_col, metric_col]].dropna()
    return {
        "metric": metric_col,
        "max_group": temp.loc[temp[metric_col].idxmax(), group_col],
        "max_value": temp[metric_col].max(),
        "min_group": temp.loc[temp[metric_col].idxmin(), group_col],
        "min_value": temp[metric_col].min(),
        "gap": temp[metric_col].max() - temp[metric_col].min()
    }


# 1) Create demo dataset
X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.55, 0.45],
    flip_y=0.03,
    random_state=RANDOM_STATE
)

# 2) Create synthetic subgroup labels
rng = np.random.default_rng(RANDOM_STATE)
race = rng.choice(["White", "Black", "Hispanic"], size=len(y), p=[0.45, 0.30, 0.25])
sex = rng.choice(["Male", "Female"], size=len(y), p=[0.52, 0.48])
age_cat = rng.choice(["18-29", "30-44", "45+"], size=len(y), p=[0.30, 0.40, 0.30])

# 3) Split data
X_train, X_test, y_train, y_test, race_train, race_test, sex_train, sex_test, age_cat_train, age_cat_test = train_test_split(
    X, y, race, sex, age_cat,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# 4) Fit baseline model
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# 5) Build audit table
audit_test = pd.DataFrame({
    "race": race_test,
    "sex": sex_test,
    "age_cat": age_cat_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 6) Overall metrics
overall_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "auc": roc_auc_score(y_test, y_prob),
    "log_loss": log_loss(y_test, y_prob),
    "positive_rate": y_pred.mean()
}

print("Overall baseline model metrics:")
for k, v in overall_metrics.items():
    print(f"{k}: {v:.4f}")

# 7) Subgroup metrics
race_metrics = compute_group_metrics(audit_test, "race")
sex_metrics = compute_group_metrics(audit_test, "sex")
age_metrics = compute_group_metrics(audit_test, "age_cat")

print("\nMetrics by race:")
print(race_metrics.round(4))

print("\nMetrics by sex:")
print(sex_metrics.round(4))

print("\nMetrics by age category:")
print(age_metrics.round(4))

# 8) Disparity summary
print("\nDisparity summary:")
for metric in ["accuracy", "auc", "positive_rate"]:
    print(f"\nRace - {metric}:")
    print(disparity_summary(race_metrics, "race", metric))

    print(f"Sex - {metric}:")
    print(disparity_summary(sex_metrics, "sex", metric))

    print(f"Age category - {metric}:")
    print(disparity_summary(age_metrics, "age_cat", metric))

Overall baseline model metrics:
accuracy: 0.8283
auc: 0.8946
log_loss: 0.4110
positive_rate: 0.4150

Metrics by race:
       race    n  accuracy  positive_rate     auc  log_loss
0     Black  177    0.8136         0.4237  0.8732    0.4504
1  Hispanic  156    0.8205         0.3974  0.8929    0.4107
2     White  267    0.8427         0.4195  0.9102    0.3851

Metrics by sex:
      sex    n  accuracy  positive_rate     auc  log_loss
0  Female  288    0.8264          0.441  0.8888    0.4278
1    Male  312    0.8301          0.391  0.8998    0.3956

Metrics by age category:
  age_cat    n  accuracy  positive_rate     auc  log_loss
0   18-29  191    0.8220         0.4293  0.8786    0.4414
1   30-44  244    0.8156         0.4098  0.8989    0.3983
2     45+  165    0.8545         0.4061  0.9088    0.3948

Disparity summary:

Race - accuracy:
{'metric': 'accuracy', 'max_group': 'White', 'max_value': 0.8426966292134831, 'min_group': 'Black', 'min_value': 0.8135593220338984, 'gap': 0.0291373071795